<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Ai/blob/main/basic_image_classification_with_pyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [497]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader ,Dataset
import numpy as np

In [498]:
torch.manual_seed(42)

In [499]:
device = torch.device("cuda")

In [500]:
df = pd.read_csv("/content/fashion-mnist_train.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [501]:
x = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [502]:
y_train = y
x_train = x/255.0
x_train.shape

(60000, 784)

Optuna

In [503]:
class customDataset(Dataset):
  def __init__(self , features , labels):
    self.features = torch.tensor(features , dtype=torch.float32).to(device)
    self.labels = torch.tensor(labels , dtype=torch.long).to(device)
  def __len__(self):
    return self.features.shape[0]
  def __getitem__(self , index):
    return self.features[index] , self.labels[index]

In [504]:
dataset_train = customDataset(x_train , y_train)
dataLoaded_train = DataLoader(dataset_train , batch_size=128 , shuffle= True)

In [505]:
df = pd.read_csv("/content/fashion-mnist_test.csv")
df.head()
x = df.iloc[:,1:].values
y = df.iloc[:,0].values
y_test = y
x_test = x/255.0
x_test.shape

(10000, 784)

In [506]:
dataset_test = customDataset(x_test , y_test)
dataLoaded_test = DataLoader(dataset_test , batch_size=128 , shuffle= True)

In [507]:
class ANN(nn.Module):
  def __init__(self):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(784 , 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p = 0.3),

        nn.Linear(128 , 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p = 0.3),

        nn.Linear(64 , 10),
    )
  def forward(self , x):
    y_pred = self.model(x)
    return y_pred

In [514]:
class MyNN(nn.Module):
  def __init__(self , input_dim , output_dim , hidden_layers , neurons_per_layers):
    super().__init__()
    layers = []
    for i in range(hidden_layers):
      layers.append(nn.Linear(input_dim , neurons_per_layers))
      layers.append(nn.BatchNorm1d(neurons_per_layers))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(0.3))
      input_dim = neurons_per_layers
    layers.append(nn.Linear(input_dim , output_dim))
    self.model = nn.Sequential(*layers)
  def forward(self , x):
    return self.model(x)

In [509]:
def objective(trial):
  num_hidden = trial.suggest_int("num_hidden_layer" , 1 , 5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer" , 8 , 128 , step = 8)

  # model init
  input_dim = 784
  output_dim = 10

  model = MyNN(input_dim , output_dim ,num_hidden , neurons_per_layer )
  model.to(device)

  epoch = 20
  loss_fn = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters() , lr=0.01 , weight_decay=1e-4)

  for i in range(epoch):
    for value , target in dataLoaded_train:
      value = value.to(device)
      target = target.to(device)
      optimizer.zero_grad()

      y_pred = model(value.float())
      loss = loss_fn(y_pred , target)

      loss.backward()
      optimizer.step()
  model.eval()
  total_train = 0
  correct_train = 0
  with torch.no_grad():
    for value , target in dataLoaded_train:

      value = value.to(device)
      target = target.to(device)

      outputs = model(value.float())
      _ , predict = torch.max(outputs , 1)
      total_train += target.shape[0]
      correct_train = correct_train + (predict == target).sum().item()
  accuracy = (correct_train/total_train)*100
  return accuracy

In [515]:
import optuna
study = optuna.create_study(direction="maximize")

[I 2026-07-06 14:51:31,867] A new study created in memory with name: no-name-47422b7e-7af6-4c0c-8665-639c3febbd4f


In [516]:
study.optimize(objective , n_trials=10)

[I 2026-07-06 14:51:58,407] Trial 0 finished with value: 85.005 and parameters: {'num_hidden_layer': 1, 'neurons_per_layer': 40}. Best is trial 0 with value: 85.005.
[I 2026-07-06 14:52:27,135] Trial 1 finished with value: 83.56833333333333 and parameters: {'num_hidden_layer': 2, 'neurons_per_layer': 16}. Best is trial 0 with value: 85.005.
[I 2026-07-06 14:53:10,301] Trial 2 finished with value: 83.93333333333334 and parameters: {'num_hidden_layer': 5, 'neurons_per_layer': 40}. Best is trial 0 with value: 85.005.
[I 2026-07-06 14:53:43,436] Trial 3 finished with value: 69.47 and parameters: {'num_hidden_layer': 3, 'neurons_per_layer': 8}. Best is trial 0 with value: 85.005.
[I 2026-07-06 14:54:26,818] Trial 4 finished with value: 80.27666666666666 and parameters: {'num_hidden_layer': 5, 'neurons_per_layer': 32}. Best is trial 0 with value: 85.005.
[I 2026-07-06 14:54:55,534] Trial 5 finished with value: 85.08 and parameters: {'num_hidden_layer': 2, 'neurons_per_layer': 128}. Best is t